# 🧪 01 - Preparación, Limpieza, Análisis Profundo y Validación de Datos
### RotBot English Coach — Dataset Engineering Pipeline

Este notebook ejecuta el análisis exploratorio de datos (EDA), la auditoría de estilo y consistencia de **RotBot** (200 ejemplos con tono sarcástico, inteligente, amigable, sin emojis y llamando 'boss' al usuario), la división estratificada y la validación formal de los archivos JSONL para Fine-Tuning.

---

## 1. Configuración de Entorno e Importaciones

In [ ]:
import os
import sys
from pathlib import Path

# Configurar directorio raíz del proyecto en sys.path
PROJECT_ROOT = Path(os.path.abspath("")).resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.parser import (
    sql_to_dataframe,
    sql_to_jsonl,
    stratified_split,
    find_default_sql_file,
    DEFAULT_SYSTEM_PROMPT
)
from src.validator import (
    validate_jsonl_dataset,
    print_dataset_summary,
    compute_detailed_token_stats,
    check_rotbot_style_compliance,
    has_emojis,
    estimate_tokens
)

print(f"✅ Proyecto cargado en: {PROJECT_ROOT}")
print(f"📌 System Prompt Rotbot:\n{DEFAULT_SYSTEM_PROMPT}")

## 2. Ingesta y Detección Automática del Dataset SQL

In [ ]:
RAW_SQL_PATH = find_default_sql_file(str(PROJECT_ROOT))
print(f"📂 Archivo SQL maestro: {RAW_SQL_PATH}")

df = sql_to_dataframe(RAW_SQL_PATH)
print(f"📊 Total de ejemplos parseados con éxito: {len(df)}")
display(df.head(5) if not df.empty else "⚠️ No se encontraron registros")

## 3. Análisis Bivariado: Categorías vs Niveles de Dificultad
Analizamos la matriz cruzada (*crosstab*) para entender el balance pedagógico de los datos.

In [ ]:
print("=== 📊 MATRIZ CRUZADA (Categoría x Nivel) ===")
ct_counts = pd.crosstab(df['category'], df['level'], margins=True, margins_name="Total")
display(ct_counts)

print("\n=== 📈 PORCENTAJES POR CATEGORÍA (% dentro de cada categoría) ===")
ct_pct = pd.crosstab(df['category'], df['level'], normalize='index') * 100
display(ct_pct.round(1))

In [ ]:
# Visualización gráfica de la distribución de categorías por nivel
try:
    category_level_df = pd.crosstab(df['category'], df['level'])
    ax = category_level_df.plot(kind='bar', stacked=True, figsize=(10, 5), colormap='viridis')
    plt.title('Distribución de Temas por Nivel de Dificultad (RotBot Dataset)', fontsize=14, pad=12)
    plt.xlabel('Categoría Pedagógica', fontsize=11)
    plt.ylabel('Cantidad de Ejemplos', fontsize=11)
    plt.xticks(rotation=25, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.legend(title='Nivel')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Nota gráfica: {e}")

## 4. Análisis Léxico y de Longitud de Tokens
Examinamos el conteo de tokens de las entradas del usuario y las respuestas de RotBot para garantizar estabilidad en el entrenamiento.

In [ ]:
# Cálculo de métricas léxicas detalladas
df['user_tokens'] = df['user_message'].astype(str).apply(estimate_tokens)
df['assistant_tokens'] = df['assistant_message'].astype(str).apply(estimate_tokens)
df['total_tokens'] = df['user_tokens'] + df['assistant_tokens']
df['user_words'] = df['user_message'].astype(str).apply(lambda x: len(x.split()))
df['assistant_words'] = df['assistant_message'].astype(str).apply(lambda x: len(x.split()))

stats_report = compute_detailed_token_stats(df)

print("=== 📏 ESTADÍSTICAS DESCRIPTIVAS DE TOKENS ===")
print(f"• Total de tokens en el dataset: {stats_report['total_tokens']['sum']:,}")
print(f"• Promedio tokens por ejemplo:    {stats_report['total_tokens']['mean']} (Mediana: {stats_report['total_tokens']['median']})")
print(f"• Rango de tokens por ejemplo:     [{stats_report['total_tokens']['min']} - {stats_report['total_tokens']['max']}]")
print(f"• Promedio tokens Usuario:        {stats_report['user_tokens']['mean']} (Mediana: {stats_report['user_tokens']['median']})")
print(f"• Promedio tokens Asistente:      {stats_report['assistant_tokens']['mean']} (Mediana: {stats_report['assistant_tokens']['median']})")
print(f"• Ratio Asistente / Usuario:      {stats_report['asst_to_user_token_ratio']}x")

In [ ]:
# Distribución visual de longitud de tokens
try:
    plt.figure(figsize=(10, 4))
    plt.hist(df['user_tokens'], bins=15, alpha=0.6, label='Usuario (User Tokens)', color='#3498db')
    plt.hist(df['assistant_tokens'], bins=15, alpha=0.6, label='RotBot (Assistant Tokens)', color='#2ecc71')
    plt.title('Histograma de Distribución de Tokens (User vs RotBot)', fontsize=13, pad=10)
    plt.xlabel('Número Estimado de Tokens')
    plt.ylabel('Frecuencia')
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Nota gráfica: {e}")

## 5. Auditoría de Personalidad y Compliance de RotBot
Verificamos rigurosamente que las respuestas cumplan las reglas fundamentales de RotBot:
1. **Uso del vocativo 'boss'.**
2. **Cero emojis en todas las respuestas.**
3. **Explicación pedagógica con tono amigable y sarcástico.**

In [ ]:
compliance = check_rotbot_style_compliance(df)

print("=== 🕵️ REPORTE DE AUDITORÍA DE ESTILO ROTBOT ===")
print(f"• Ejemplos evaluados:            {compliance['total_evaluated']}")
print(f"• Presencia del vocativo 'boss': {compliance['boss_count']}/{compliance['total_evaluated']} ({compliance['boss_compliance_pct']}%)")
print(f"• Violaciones de emojis:         {compliance['emoji_violations_count']} (Libre de emojis: {compliance['emoji_free_pct']}%)")
print(f"• Estado de Compliance:          {'✅ 100% COMPLIANT' if compliance['is_fully_compliant'] else '⚠️ REVISAR CASOS'}")

In [ ]:
# Inspección cualitativa de ejemplos por categoría
print("=== 🔍 MUESTRAS REPRESENTATIVAS POR CATEGORÍA ===")
for cat in df['category'].unique():
    sample = df[df['category'] == cat].iloc[0]
    print("=" * 60)
    print(f"📁 CATEGORÍA: {cat.upper()} | Nivel: {sample.get('level', 'N/A')}")
    print(f"👤 Usuario:   \"{sample['user_message']}\"")
    print(f"🤖 RotBot:    \"{sample['assistant_message']}\"")
    if pd.notna(sample.get('notes')):
        print(f"🎯 Error Map: {sample['notes']}")

## 6. División Estratificada Multivariable (80% Train / 20% Val)
Utilizamos `stratified_split` para garantizar que las proporciones de `category` y `level` sean idénticas tanto en el conjunto de entrenamiento como en el de validación.

In [ ]:
TRAIN_JSONL_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "train.jsonl")
VAL_JSONL_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "val.jsonl")

# Exportar con estratificación balanceada
train_count, val_count = sql_to_jsonl(
    sql_path=RAW_SQL_PATH,
    output_train_path=TRAIN_JSONL_PATH,
    output_val_path=VAL_JSONL_PATH,
    val_ratio=0.2,
    stratify=True,
    format_type="chatml",
    seed=42
)

print(f"✅ Exportación Estratificada Exitosa:")
print(f"  - 🏋️ Train: {train_count} ejemplos (80%) -> {TRAIN_JSONL_PATH}")
print(f"  - 🧪 Val:   {val_count} ejemplos (20%) -> {VAL_JSONL_PATH}")

## 7. Reporte Final de Calidad y Validación para Fine-Tuning

In [ ]:
print("--- Validando Train Dataset ---")
train_report = validate_jsonl_dataset(TRAIN_JSONL_PATH)
print_dataset_summary(train_report, title="Reporte de Validación - Train Dataset")

print("\n--- Validando Validation Dataset ---")
val_report = validate_jsonl_dataset(VAL_JSONL_PATH)
print_dataset_summary(val_report, title="Reporte de Validación - Validation Dataset")